# Import modules

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

# Import Multi-task ElasticNet

In [ ]:
import joblib

model_1 = joblib.load('ElasticNet.pkl')

In [ ]:
model_features = model_1.feature_names_in_
model_features

In [ ]:
import pickle

with open('y_train_features.pkl', 'rb') as f:
    y_train_features_loaded = pickle.load(f)

print(y_train_features_loaded)

# Import Random forest regressor

In [ ]:
model_2 = joblib.load('Random Forest Regressor.pkl')

# Import ensemble

In [ ]:
final_estimator = joblib.load('Ensemble.pkl')

# Microarray data

In [ ]:
files_dict = {
'GSE26193_gene_expression.csv' : 'GSE26193_metabolomics.csv',
'GSE62452_gene_expression.csv' : 'GSE62452_metabolomics.csv',
'GSE37751_gene_expression.csv' : 'GSE37751_metabolomics.csv',
'GSE89076_gene_expression.csv' : 'GSE89076_metabolomics.csv',
'Cornell_PROSTATE_gene_expression.csv' : 'Cornell_PROSTATE_metabolomics.csv',
'Multiregions_gene_expression.csv' : 'Multiregions_metabolomics.csv',
'GSE76297_gene_expression.csv' : 'GSE76297_metabolomics.csv'
}

In [ ]:
path = 'Microarrays/'
results_path = 'Microarray_res'

# Functions

In [ ]:
def filter_model_features(df):
  filtered_data = df[model_features]
  return filtered_data

In [ ]:
def predict_metabolites(model, filtered_data):
  pred_metabolites = model.predict(filtered_data)
  pred_metabolites = pd.DataFrame(pred_metabolites, index = filtered_data.index, columns = y_train_features_loaded)
  return pred_metabolites

In [ ]:
def predict_metabolites_ensemble(model_1, model_2, final_estimator, filtered_data):
  pred_1 = model_1.predict(filtered_data)
  pred_2 = model_2.predict(filtered_data)
  stacked_preds = np.hstack([pred_1, pred_2])
  pred_metabolites = final_estimator.predict(stacked_preds)
  pred_metabolites = pd.DataFrame(pred_metabolites, index = filtered_data.index, columns = y_train_features_loaded)

  return pred_metabolites

In [ ]:
from scipy.stats import spearmanr

metabolites_spearman_positive_all = {}
metabolites_spearman_negative_all = {}

def plot_spearman_per_metabolite(y_pred, y_true, model_name, rna_file_name):
    """Plots the Spearman's correlation coefficient per metabolite."""

    spearman_coeffs = []
    for i in range(y_true.shape[1]):
        coeff, _ = spearmanr(y_true.iloc[:, i], y_pred.iloc[:, i])
        spearman_coeffs.append(coeff)

    spearman_df = pd.DataFrame({'Metabolite': y_true.columns, 'Spearman Coefficient': spearman_coeffs})

    spearman_df = spearman_df.sort_values(by=['Spearman Coefficient'], ascending=False)

    dict_pos = {}
    metabolites_spearman_positive = spearman_df[spearman_df['Spearman Coefficient'] > 0]
    dict_pos[rna_file_name] =  metabolites_spearman_positive['Metabolite'].to_list()
    metabolites_spearman_positive_all.setdefault(model_name, {}).update(dict_pos)

    dict_neg = {}
    metabolites_spearman_negative = spearman_df[spearman_df['Spearman Coefficient'] < 0]
    dict_neg[rna_file_name] =  metabolites_spearman_negative['Metabolite'].to_list()
    metabolites_spearman_negative_all.setdefault(model_name, {}).update(dict_neg)

    plt.figure(figsize=(10, 6))
    plt.title(f"Spearman's Correlation Coefficient per Metabolite ({model_name} {rna_file_name})")
    plt.bar(range(len(spearman_df)), spearman_df['Spearman Coefficient'])
    plt.xticks()
    plt.xlabel("Metabolite")
    plt.savefig(f'{results_path}/Spearmans Corr per metabolite({model_name} {rna_file_name}).svg', bbox_inches = 'tight')
    plt.show()

In [ ]:
def get_spearman_correlation(y_pred, y_true, model_name, rna_file_name):

  correlation, p_value = spearmanr(y_true, y_pred)

  correlation_df = pd.DataFrame({'Actual': y_true.values.ravel(), 'Predicted': y_pred.values.ravel()})

  correlation, p_value = spearmanr(correlation_df['Actual'], correlation_df['Predicted'])

  correlation_df['Correlation'] = correlation
  correlation_df['P-value'] = p_value

  sns.regplot(x='Actual', y='Predicted', data=correlation_df, line_kws={'color': 'red'})
  plt.title(f'Spearman\'s Correlation ({model_name} {rna_file_name})')
  plt.xlabel('Actual Values')
  plt.ylabel('Predicted Values')

  plt.text(0.1, 0.9, f'Correlation: {correlation:.2f}\nP-value: {p_value:.2f}', transform=plt.gca().transAxes)
  plt.savefig(f'{results_path}/Spearmans Corr ({model_name} {rna_file_name}).svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
def analyse_microarray(rna_file, lcms_file):

  rna_file_name = rna_file.split('_')[0]

  print('-------------------------------------')
  print('                                     ')
  print(f'Analysing {rna_file_name} !')
  print('                                     ')
  print('-------------------------------------')

  microarray = pd.read_csv(path + rna_file, index_col=0)
  rna_filtered = filter_model_features(microarray)

  #Elastic Net predictions
  rna_pred_elastic = predict_metabolites(model_1, rna_filtered)

  #Random forest regressor predictions
  rna_pred_rf = predict_metabolites(model_2, rna_filtered)

  #Ensemble predictions
  rna_pred_ensemble = predict_metabolites_ensemble(model_1, model_2, final_estimator, rna_filtered)

  #LCMS data
  lcms = pd.read_csv(path + lcms_file, index_col=0)

  #Make sure the columns are the same
  lcms = lcms[list(set(lcms.columns).intersection(set(rna_pred_elastic.columns)))]
  rna_pred_elastic = rna_pred_elastic[list(set(lcms.columns).intersection(set(rna_pred_elastic.columns)))]
  rna_pred_rf = rna_pred_rf[list(set(lcms.columns).intersection(set(rna_pred_rf.columns)))]
  rna_pred_ensemble = rna_pred_ensemble[list(set(lcms.columns).intersection(set(rna_pred_ensemble.columns)))]

  #Make sure the indexes are the same
  lcms = lcms.loc[list(set(lcms.index).intersection(set(rna_pred_elastic.index)))]
  rna_pred_elastic = rna_pred_elastic.loc[list(set(lcms.index).intersection(set(rna_pred_elastic.index)))]
  rna_pred_rf = rna_pred_rf.loc[list(set(lcms.index).intersection(set(rna_pred_rf.index)))]
  rna_pred_ensemble = rna_pred_ensemble.loc[list(set(lcms.index).intersection(set(rna_pred_ensemble.index)))]

  #Get plots

  #Spearman per metabolite
  plot_spearman_per_metabolite(y_pred = rna_pred_elastic, y_true = lcms, model_name = 'Elastic Net', rna_file_name = rna_file_name)
  plot_spearman_per_metabolite(y_pred = rna_pred_rf, y_true = lcms, model_name = 'Random Forest Regressor', rna_file_name = rna_file_name)
  plot_spearman_per_metabolite(y_pred = rna_pred_ensemble, y_true = lcms, model_name = 'Ensemble', rna_file_name = rna_file_name)

  #Spearman for the whole dataset
  get_spearman_correlation(y_pred = rna_pred_elastic, y_true = lcms, model_name = 'Elastic Net', rna_file_name = rna_file_name)
  get_spearman_correlation(y_pred = rna_pred_rf, y_true = lcms, model_name = 'Random Forest Regressor', rna_file_name = rna_file_name)
  get_spearman_correlation(y_pred = rna_pred_ensemble, y_true = lcms, model_name = 'Ensemble', rna_file_name = rna_file_name)


  print('-------------------------------------')
  print('                                     ')
  print(f'Analysis done for {rna_file_name} !')
  print('                                     ')
  print('-------------------------------------')

# Analysis

In [ ]:
for item, value in files_dict.items():
  analyse_microarray(rna_file = item, lcms_file = value)

In [ ]:
with open(results_path + 'metabolites_spearman_positive_all.pkl', 'wb') as file:
    pickle.dump(metabolites_spearman_positive_all, file)

In [ ]:
with open(results_path + 'metabolites_spearman_negative_all.pkl', 'wb') as file:
    pickle.dump(metabolites_spearman_negative_all, file)

In [ ]:
metabolites_spearman_positive_all

In [ ]:
metabolites_spearman_negative_all

In [ ]:
def plot_model(model_name):
  df_pos = metabolites_spearman_positive_all[model_name]
  df_neg = metabolites_spearman_negative_all[model_name]

  positive = {}
  for item in df_pos:
    positive[item] = len(df_pos[item])
    print(item, len(df_pos[item]))
  positive = pd.DataFrame(positive, index = [0])
  positive.rename(index = {0 : 'Positive'}, inplace = True)

  negative = {}
  for item in df_neg:
    negative[item] = len(df_neg[item])
    print(item, len(df_neg[item]))
  negative = pd.DataFrame(negative, index = [0])
  negative.rename(index = {0 : 'Negative'}, inplace = True)

  pos_neg = pd.concat([positive, negative], axis = 0)
  pos_neg = pos_neg.T

  pos_neg = pos_neg.drop('GSE37751')
  pos_neg = pos_neg.sort_values(by = 'Positive', ascending = False)

  pos_neg.plot(kind = 'bar', stacked = True)
  plt.legend(bbox_to_anchor = (1.05, 1), loc = 'upper left')
  plt.title(f"Microarrays Spearman's correlation prediction vs actual ({model_name})")
  plt.ylabel('Counts')
  plt.savefig(f'{results_path}/Microarrays_{model_name}.svg', bbox_inches = 'tight')
  plt.show()

In [ ]:
plot_model('Elastic Net')
plot_model('Random Forest Regressor')
plot_model('Ensemble')